In [1]:
from pathlib import Path
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")

DATA = Path("../../data")  # notebook nằm ở Python/Week 2/, lùi 2 cấp như Ngày 3
print(DATA.resolve())
print(sorted(p.name for p in DATA.glob("weather.*")))

D:\BI\projects\data-pipeline-project\data
['weather.csv', 'weather.json', 'weather.parquet']


In [2]:
table_id = f"{PROJECT}.raw.weather_parquet_py"
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",
)
with open(DATA / "weather.parquet", "rb") as f:
    job = client.load_table_from_file(f, table_id, job_config=job_config)
job.result()  # chờ job xong

print(job.job_id, job.state, job.output_rows)
t = client.get_table(table_id)
print([(s.name, s.field_type) for s in t.schema])

70d98785-a4f2-4e05-839a-1dffce4f8e0f DONE 21
[('time', 'TIMESTAMP'), ('temperature_2m_max', 'FLOAT'), ('temperature_2m_min', 'FLOAT'), ('precipitation_sum', 'FLOAT'), ('city', 'STRING'), ('ingested_at', 'TIMESTAMP')]


In [3]:
print(open(DATA / "weather.csv", encoding="utf-8").readline())

time,temperature_2m_max,temperature_2m_min,precipitation_sum,city,ingested_at



In [4]:
schema = [
    bigquery.SchemaField("time", "DATE"),
    bigquery.SchemaField("temperature_2m_max", "FLOAT"),
    bigquery.SchemaField("temperature_2m_min", "FLOAT"),
    bigquery.SchemaField("precipitation_sum", "FLOAT"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]
table_id = f"{PROJECT}.raw.weather_csv_py"
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,   # bỏ dòng tiêu đề
    schema=schema,
    write_disposition="WRITE_TRUNCATE",
)
with open(DATA / "weather.csv", "rb") as f:
    job = client.load_table_from_file(f, table_id, job_config=job_config)
job.result()
print(job.state, job.output_rows)
print([(s.name, s.field_type) for s in client.get_table(table_id).schema])

DONE 21
[('time', 'DATE'), ('temperature_2m_max', 'FLOAT'), ('temperature_2m_min', 'FLOAT'), ('precipitation_sum', 'FLOAT'), ('city', 'STRING'), ('ingested_at', 'TIMESTAMP')]


In [5]:
uri = "gs://cloud-samples-data/bigquery/us-states/us-states.parquet"
table_id = f"{PROJECT}.raw.us_states"
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",
)
job = client.load_table_from_uri(uri, table_id, job_config=job_config)
job.result()
print(job.state, job.output_rows)
print([(s.name, s.field_type) for s in client.get_table(table_id).schema])

DONE 50
[('name', 'STRING'), ('post_abbr', 'STRING')]


In [ ]:
print(sorted(t.table_id for t in client.list_tables(f"{PROJECT}.raw")))